In [3]:
from transformers import (
    TokenClassificationPipeline,
    AutoModelForTokenClassification,
    AutoTokenizer,
)
from transformers.pipelines import AggregationStrategy
import numpy as np

import warnings
warnings.filterwarnings("error")

## Define Pipeline

https://huggingface.co/ml6team/keyphrase-extraction-kbir-inspec

In [4]:
class KeyphraseExtractionPipeline(TokenClassificationPipeline):
    def __init__(self, model, *args, **kwargs):
        super().__init__(
            model=AutoModelForTokenClassification.from_pretrained(model),
            tokenizer=AutoTokenizer.from_pretrained(model),
            *args,
            **kwargs
        )

    def postprocess(self, all_outputs):
        results = super().postprocess(
            all_outputs=all_outputs,
            aggregation_strategy=AggregationStrategy.SIMPLE,
        )
        return np.unique([result.get("word").strip() for result in results])

In [5]:
model_name = "ml6team/keyphrase-extraction-kbir-inspec"
extractor = KeyphraseExtractionPipeline(model=model_name)

In [8]:
text = """
Transparency has emerged as a critical issue in the field of artificial intelligence (AI) as concerns around the interpretability, explainability, and accountability of these systems become more prevalent.
One of the main concerns around AI systems is that it can be difficult for humans to understand how decisions are made. This lack of transparency can have significant ethical implications when these systems are used in high-stakes decision-making contexts such as healthcare. We will provide a brief overview of the history of AI and how it has evolved, including the factors that contributed to the current black-box nature of many AI systems. Then, we will examine the use of AI in various industries including healthcare and companies like Amazon and Twitter. We will highlight the ethical risks and potential and actual consequences of a lack of transparency. We will also review existing literature and technologies revolving around AI transparency challenges and solutions including various model interpretation methods and tools that have been developed to address the interpretability and explainability of AI systems. Additionally, we will discuss the effects it has on privacy.
We hope to provide valuable insights for individual users, companies, and AI researchers about the importance of transparency in AI systems.
""".replace("\n", " ")

keyphrases = extractor(text)
keyphrases

array(['Amazon', 'Transparency', 'Twitter', 'accountability',
       'artificial intelligence', 'ethical', 'explainability',
       'healthcare', 'history', 'interpretability', 'privacy',
       'transparency'], dtype='<U23')

# Semantic Scholar

Rate limit:
1 request per second for the following endpoints:
/paper/batch
/paper/search
/recommendations
10 requests / second for all other calls

In [17]:
import requests
import os
import sys

In [18]:
def print_papers(papers):
    for idx, paper in enumerate(papers):
        print(f"{idx}  {paper['title']} {paper['url']}")


def get_papers(search_words, result_limit):
    query_words = '+'.join(search_words)
    url = f'http://api.semanticscholar.org/graph/v1/paper/search'
    rsp = requests.get(url,
                        headers={'X-API-KEY': os.getenv('S2_API_KEY')},
                        params={'query': query_words, 'limit':result_limit, 'fields':'title,authors,url'})
    rsp.raise_for_status()
    results = rsp.json()

    total = results["total"]
    if not total:
        raise 'No matches found. Please try another query.'
        sys.exit()
        
    print(f'Found {total} results. Showing up to {result_limit}.')
    return results['data']

result_limit = 10
get_papers(['Amazon', 'Transparency', 'Twitter', 'accountability',
       'artificial intelligence', 'ethical', 'explainability',
       'healthcare', 'history', 'interpretability', 'privacy',
       'transparency'], result_limit)

Found 2 results. Showing up to 10.


[{'paperId': '3602a1acbad352baafedaf8bea10675e9027d334',
  'url': 'https://www.semanticscholar.org/paper/3602a1acbad352baafedaf8bea10675e9027d334',
  'title': 'Decoding the Black Box: A Comprehensive Review of Explainable Artificial Intelligence',
  'authors': [{'authorId': '8003685', 'name': 'Ossama H. Embarak'}]},
 {'paperId': '7dfa7d32d8ffa777095e6aa56aa629bc80742dd1',
  'url': 'https://www.semanticscholar.org/paper/7dfa7d32d8ffa777095e6aa56aa629bc80742dd1',
  'title': 'Artificial Intelligence in Medicine: Revolutionizing Healthcare for Improved Patient Outcomes',
  'authors': [{'authorId': '38680767', 'name': 'Varshil Mehta'}]}]

# Function

In [20]:
def text_recommendations(text):
    text = text.replace("\n", " ")
    keyphrases = extractor(text)
    print("Keyphrases:", keyphrases)
    papers = get_papers(keyphrases, result_limit=10)
    print_papers(papers)

In [21]:
# Paper: A Contextual Latent Space Model: Subsequence Modulation in Melodic Sequence
abstract = "Some generative models for sequences such as music and text allow us to edit only subsequences, given surrounding context sequences, which plays an important part in steering generation interactively. However, editing subsequences mainly involves randomly resampling subsequences from a possible generation space. We propose a contextual latent space model (CLSM) in order for users to be able to explore subsequence generation with a sense of direction in the generation space, e.g., interpolation, as well as exploring variations—semantically similar possible subsequences. A context-informed prior and decoder constitute the generative model of CLSM, and a context position-informed encoder is the inference model. In experiments, we use a monophonic symbolic music dataset, demonstrating that our contextual latent space is smoother in interpolation than baselines, and the quality of generated samples is superior to baseline models. The generation examples are available online."
text_recommendations(abstract)

Keyphrases: ['CL' 'CLSM' 'contextual latent space model' 'generative model'
 'generative models' 'inference model' 'interpolation'
 'monophonic symbolic music' 'music']
Found 1 results. Showing up to 10.
0  A Contextual Latent Space Model: Subsequence Modulation in Melodic Sequence https://www.semanticscholar.org/paper/36d1aeff3f1e57f2f2bf3cd4f596d7797862bc86
